# Unigram Model

Based on the pseudocode provided on the slides 26-27 from https://www.phontron.com/slides/nlp-programming-en-01-unigramlm.pdf

In [1]:
from collections import defaultdict  
import math

In [6]:
training_sentences = [
    "I like tea",
    "I am tired",
    "I am tired",
    "The dog barks",
    "I am happy"
]

test_sentences = [
    "I like coffee",
    "The dog sleeps",
    "She drinks tea",  # contains unseen word "drinks"
]

In [4]:
def train_unigram(training_sentences):
    """
    Trains a unigram model from a list of sentences.
    Returns a dictionary with each word's probability.
    """
    # defaultdict(int) creates a dictionary where missing keys automatically start at 0. 
    # We can do counts[word] += 1 without checking if 'word' already exists.
    
    counts = defaultdict(int)
    
    total_count = 0  # total number of word tokens (including </s>)
    
    # Go through each sentence in the training data
    for line in training_sentences:
        words = line.strip().split()   # split the line into individual words
        words.append("</s>")           # append an end-of-sentence marker
        
        for word in words:
            counts[word] += 1          # increase count for this word
            total_count += 1           # increase total word count
    
    # Compute probabilities for each word
    # P(word) = count(word) / total_count
    model = {word: counts[word] / total_count for word in counts}
    
    return model

In [5]:
def test_unigram(model, test_sentences, lambda_unk=0.05):
    """
    Tests a trained unigram model on test sentences.
    Computes entropy (average negative log probability)
    and coverage (fraction of words seen during training).
    
    lambda_unk: smoothing constant for unknown words
    """
    
    V = len(model)  # vocabulary size
    H = 0           # total entropy sum
    W = 0           # total number of words seen
    unk = 0         # count of unknown words
    
    for line in test_sentences:
        words = line.strip().split()
        words.append("</s>")
        
        for w in words:
            W += 1
            
            # Start with probability for unknown words
            P = lambda_unk / V  
            
            # If the word exists in the model, add its weighted probability
            if w in model:
                P += (1 - lambda_unk) * model[w]
            else:
                unk += 1  # unseen word
            
            # Add -log2(P) to entropy sum
            H += -math.log2(P)
    
    # Compute final entropy and coverage
    entropy = H / W
    coverage = (W - unk) / W
    
    return entropy, coverage

In [10]:
# Train the unigram model
unigram_model = train_unigram(training_sentences)

# Show the learned probabilities
print("Unigram Model (word probabilities):")
for word, prob in unigram_model.items():
    print(f"{word:10s} → {prob:.4f}")

# Evaluate the model
entropy, coverage = test_unigram(unigram_model, test_sentences)

# Print results
print(" ")
print("Evaluation Results:")
print(f"Entropy  = {entropy:.4f}")
print(f"Coverage = {coverage:.4f}")

Unigram Model (word probabilities):
I          → 0.2000
like       → 0.0500
tea        → 0.0500
</s>       → 0.2500
am         → 0.1500
tired      → 0.1000
The        → 0.0500
dog        → 0.0500
barks      → 0.0500
happy      → 0.0500
 
Evaluation Results:
Entropy  = 4.6727
Coverage = 0.6667
